In [25]:
from fractions import Fraction
from typing import Tuple, Union
import numpy as np
import pandas as pd
from sympy import Matrix
from IPython.display import display

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.float_format', '{:.15f}'.format)  # Ép hiển thị dạng thập phân chuẩn
pd.set_option('display.width', 300)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp viền quanh tìm ma trận nghịch đảo

# Thuật toán
**Input**: $A_{n \times n}$.

**Output**: $A^{-1}$

1. Khởi tạo ma trận $M = A^T A$. Đặt $M_k$ là ma trận con tạo bởi $k$ hàng đầu và $k$ cột đầu của $M$\, $B_k = M_k^{-1}$

2. Xét $M_1 = [m_{1,1}]$:

    - Nếu $m_{1,1} = 0 \Rightarrow $ Báo không tồn tại $A^{-1}$, dừng chương trình.

    - Trái lại, $B_1 = \left[\dfrac{1}{m_{1,1}}\right]$

3. Lặp $k$ từ $2$ đến $n$:

    3.1. Đặt $M_k = \begin{bmatrix} M_{k-1} & \alpha_{k-1, 1} \\ \alpha_{1, k-1} & m_{k,k} \end{bmatrix}$, trong đó:

      $\alpha_{k-1, 1} = \begin{bmatrix} m_{1,k} & m_{2,k} & \dots & m_{k-1,k} \end{bmatrix}^{T}$

      $\alpha_{1, k-1} = \begin{bmatrix} m_{k,1} & m_{k,2} & \dots & m_{k,k-1} \end{bmatrix}$

    3.2. Tính $\theta = m_{k,k} - \alpha_{1, k-1} \cdot B_{k-1} \cdot \alpha_{k-1, 1}$

    3.3. Nếu $\theta = 0 \Rightarrow$ Báo không tồn tại $A^{-1}$, dừng chương trình.

    3.4. Thực hiện lần lượt các phép gán:

    - $b_{k,k} = \dfrac{1}{\theta}$


    - $\beta_{k-1, 1} = \dfrac{- B_{k-1} \cdot \alpha_{k-1, 1}}{\theta}$


    - $\beta_{1, k-1} = \dfrac{- \alpha_{1, k-1} \cdot B_{k-1}}{\theta}$


    - $B_{k-1} = B_{k-1}\left( I_{k-1} + \alpha_{k-1, 1} \cdot \beta_{1, k-1} \right)$


    - $B_k = \begin{bmatrix} B_{k-1} & \beta_{k-1, 1} \\ \beta_{1, k-1} & b_{k,k} \end{bmatrix}$

4. Sau khi lặp, thu được $M^{-1} = B_n$. Tính và in ra $A^{-1} = M^{-1} A^T$

In [ ]:
def input_matrix(filename, convert_fractions=False):
    """
    Reads a matrix from a text file and returns it as a NumPy array.
    Supports fractional entries if present (though your files use decimals).
    """
    matrix = []

    with open(filename, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue

            row = []
            for token in tokens:
                if '/' in token:
                    # convert fractions if any includes fraction sign
                    val = Fraction(token)
                    row.append(float(val) if convert_fractions else val)
                else:
                    # parse as float directly
                    row.append(float(token))

            matrix.append(row)
            
    dtype = float if convert_fractions else object
    return np.array(matrix, dtype=dtype)


In [ ]:
def output_matrix(X: np.ndarray, precision: int = 7):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    # Render natively as a LaTeX matrix in Jupyter
    # Ép kiểu về float để tránh lỗi np.round với mảng dtype=object
    X_float = np.array(X, dtype=float)
    display(Matrix(np.round(X_float, precision)))

In [ ]:
def compute_theta(A: np.ndarray, B_prev: np.ndarray, k: int) -> float:
    # Theo slide: theta = a_{kk} - alpha_{1, k-1} * B_{k-1} * alpha_{k-1, 1}
    alpha_row = A[k-1, :k-1] # alpha_{1, k-1}
    alpha_col = A[:k-1, k-1] # alpha_{k-1, 1}
    a_kk = A[k-1, k-1]
    
    theta = float(a_kk - alpha_row @ B_prev @ alpha_col)
    if np.isclose(theta, 0):
        raise ValueError(f"Ma trận suy biến: Tồn tại định thức con bằng 0 (θ_{k} = 0 tại bước {k}).")
    return theta

In [ ]:
def compute_b_nn(A: np.ndarray, B_prev: np.ndarray, k: int) -> float:
    # Theo slide: b_{kk} = 1 / theta
    theta = compute_theta(A, B_prev, k)
    return 1.0 / theta

In [ ]:
def compute_beta_col(A: np.ndarray, B_prev: np.ndarray, k: int) -> np.ndarray:
    # Theo slide: beta_{k-1, 1} = - B_{k-1} * alpha_{k-1, 1} / theta
    alpha_col = A[:k-1, k-1]
    theta = compute_theta(A, B_prev, k)
    return -B_prev.dot(alpha_col) / theta

In [ ]:
def compute_beta_row(A: np.ndarray, B_prev: np.ndarray, k: int) -> np.ndarray:
    # Theo slide: beta_{1, k-1} = - alpha_{1, k-1} * B_{k-1} / theta
    alpha_row = A[k-1, :k-1]
    theta = compute_theta(A, B_prev, k)
    return -alpha_row.dot(B_prev) / theta

In [ ]:
def update_top_left_block(A: np.ndarray, B_prev: np.ndarray, k: int, beta_row: np.ndarray) -> np.ndarray:
    # Theo slide (8): B_{k-1}^{new} = B_{k-1} @ (E_{k-1} - ... ) = B_{k-1} @ (E_{k-1} + alpha_{k-1, 1} (outer) beta_{1, k-1})
    alpha_col = A[:k-1, k-1]
    E = np.eye(k-1)
    # outer(alpha_col, beta_row) chính là tích cột x hàng tạo ra ma trận
    return B_prev.dot(E + np.outer(alpha_col, beta_row))

In [ ]:
def block_matrix_recursion(A: np.ndarray) -> np.ndarray:
    """
    Computes A^{-1} via block recursive inversion on leading principal minors.
    """
    n = A.shape[0]
    # Base case k=1
    theta1 = A[0,0]
    if np.isclose(theta1, 0):
        raise ValueError("Ma trận suy biến: m_11 = 0.")
    B_prev = np.array([[1.0/theta1]])
    # Recursively build inverse for k=2..n
    for k in range(2, n+1):
        print(f"Theta {k}:", compute_theta(A, B_prev, k))
        # Compute new blocks
        beta_col = compute_beta_col(A, B_prev, k)
        beta_row = compute_beta_row(A, B_prev, k)
        b_kk = compute_b_nn(A, B_prev, k)
        # Update top-left (k-1)x(k-1)
        B_tl = update_top_left_block(A, B_prev, k, beta_row)
        # Assemble B_k
        Bk = np.zeros((k, k))
        Bk[:k-1, :k-1] = B_tl
        Bk[:k-1, k-1] = beta_col
        Bk[k-1, :k-1] = beta_row
        Bk[k-1, k-1] = b_kk
        B_prev = Bk
        print(f"Size {k}:")
        output_matrix(B_prev)
    return B_prev

In [ ]:
def inverse_via_ata(A: np.ndarray) -> np.ndarray:
    """
    Computes A^{-1} indirectly by inverting M = A^T A via block recursion,
    then A^{-1} = M^{-1} A^T.
    """
    if A.shape[0] != A.shape[1]:
        raise ValueError("Ma trận A phải là ma trận vuông.")
        
    M = A.T.dot(A)
    M_inv = block_matrix_recursion(M)
    return M_inv.dot(A.T)

## Kết quả

In [ ]:
#Original matrix Ax=B
A = input_matrix('BLMT_input_A.txt', convert_fractions=False)

print("\nMatrix A:"); output_matrix(A)
print("\nMatrix M = A^T * A:"); output_matrix(A.T.dot(A))


Matrix A:


Matrix([
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],
[2.0, 4.0, 1.0, 5.0, 6.0, 7.0, 8.0],
[3.0, 1.0, 5.0, 2.0, 7.0, 8.0, 9.0],
[4.0, 5.0, 2.0, 6.0, 8.0, 9.0, 1.0],
[5.0, 6.0, 7.0, 8.0, 9.0, 1.0, 2.0],
[6.0, 7.0, 8.0, 9.0, 1.0, 2.0, 3.0],
[7.0, 8.0, 9.0, 1.0, 2.0, 3.0, 4.0]])


Matrix M = A^T * A:


Matrix([
[140.0, 161.0, 174.0, 145.0, 135.0, 118.0, 110.0],
[161.0, 195.0, 195.0, 179.0, 158.0, 137.0, 125.0],
[174.0, 195.0, 233.0, 176.0, 161.0, 133.0, 150.0],
[145.0, 179.0, 176.0, 227.0, 195.0, 158.0, 139.0],
[135.0, 158.0, 161.0, 195.0, 260.0, 217.0, 183.0],
[118.0, 137.0, 133.0, 158.0, 217.0, 244.0, 199.0],
[110.0, 125.0, 150.0, 139.0, 183.0, 199.0, 224.0]])

In [ ]:
A_inv = inverse_via_ata(A)
print("Inverse of A result:")
output_matrix(A_inv)

Theta 2: 9.850000000000023
Size 2:


Matrix([
[-0.1271211, -0.1167513],
[-0.1167513,  0.1015228]])

Theta 3: 8144.053517041316
Size 3:


Matrix([
[-0.3745058, -0.1196049, 0.0055115],
[-0.1196049,  0.1014899,   6.36e-5],
[ 0.0055115,    6.36e-5, 0.0001228]])

Theta 4: 10768.723288363477
Size 4:


...

Matrix([
[-3.9412242, -0.1002378,  0.0454567,  0.0278775,  0.0151204, 0.0086905],
[-0.1002378,  0.1013646, -0.0001531, -0.0001708,   -6.85e-5,  -3.73e-5],
[ 0.0454567, -0.0001531, -0.0003246,  -0.000312, -0.0001695,  -9.74e-5],
[ 0.0278775, -0.0001708,  -0.000312,    -5.1e-5, -0.0001039,  -5.97e-5],
[ 0.0151204,   -6.85e-5, -0.0001695, -0.0001039,     2.7e-5,  -3.25e-5],
[ 0.0086905,   -3.73e-5,   -9.74e-5,   -5.97e-5,   -3.25e-5,   3.72e-5]])

Theta 7: 45805.40971422771
Size 7:


Matrix([
[   -7.9943, -0.0853964,   0.090972,  0.0557249,   0.030296,  0.0174371,  0.0094066],
[-0.0853964,  0.1013102, -0.0003198, -0.0002727, -0.0001241,   -6.93e-5,   -3.44e-5],
[  0.090972, -0.0003198, -0.0008357, -0.0006247, -0.0003399, -0.0001956, -0.0001056],
[ 0.0557249, -0.0002727, -0.0006247, -0.0002423, -0.0002081, -0.0001198,   -6.46e-5],
[  0.030296, -0.0001241, -0.0003399, -0.0002081,   -2.98e-5,   -6.53e-5,   -3.52e-5],
[ 0.0174371,   -6.93e-5, -0.0001956, -0.0001198,   -6.53e-5,    1.83e-5,   -2.03e-5],
[ 0.0094066,   -3.44e-5, -0.0001056,   -6.46e-5,   -3.52e-5,   -2.03e-5,    2.18e-5]])

Inverse of A result:


Matrix([
[-7.3473279,  -15.5815, -23.0657578, -31.4791796, -39.0923608, -47.2408845, -55.6182687],
[ 0.1138963, 0.2312593,  -0.1587565,   0.1610384,   0.1752041,   0.1914142,   0.2089623],
[ 0.0817135, 0.1724515,   0.2622728,   0.3522835,   0.4386274,   0.5302374,   0.6244103],
[ 0.0501244, 0.1059184,    0.160297,   0.2160251,   0.2685546,   0.3246205,   0.3809938],
[ 0.0274083, 0.0577976,   0.0876002,   0.1177739,   0.1462873,   0.1760488,   0.2074156],
[ 0.0158738, 0.0333765,   0.0505312,   0.0679142,    0.083832,   0.1014047,   0.1194679],
[ 0.0086172, 0.0180679,   0.0273155,   0.0364126,   0.0452764,   0.0547616,   0.0645115]])